In [1]:
import sqlite3, pandas as pd
import os
conn = sqlite3.connect('../sql/shop.db')

In [2]:
pd.read_sql("""
    SELECT id, customer_id, ROUND(amount) AS amount,
           LAG(amount) OVER (PARTITION BY customer_id ORDER BY order_date) AS prev
    FROM orders WHERE customer_id = 6 ORDER BY order_date
""", conn)

,id,customer_id,amount,prev
0,31,6,5580.0,NaN
1,24,6,2079.0,5580.436067
2,20,6,4428.0,2079.091205
3,26,6,7081.0,4427.803997
4,11,6,9710.0,7081.137248
5,15,6,7473.0,9709.910442
6,37,6,5615.0,7473.145343
7,7,6,8245.0,5614.618308
8,21,6,3192.0,8245.339971
9,4,6,7882.0,3192.429750


prev_amount - это цена предыдущего заказа по времени у самого первого заказа нету предыдущего, поэтому неизвестно что там а раз неизвестно то - NULL

в следющем коде посчитаю разницу между настоящим и предыдущим и запишу в столбец difference

In [3]:
pd.read_sql("""
    SELECT id, customer_id, ROUND(amount) AS amount,
           LAG(amount) OVER (PARTITION BY customer_id ORDER BY order_date) AS prev,
           ROUND(amount) - LAG(amount) OVER (PARTITION BY customer_id ORDER BY order_date) AS difference
    FROM orders WHERE customer_id = 6 ORDER BY order_date
""", conn)

,id,customer_id,amount,prev,difference
0,31,6,5580.0,NaN,NaN
1,24,6,2079.0,5580.436067,-3501.436067
2,20,6,4428.0,2079.091205,2348.908795
3,26,6,7081.0,4427.803997,2653.196003
4,11,6,9710.0,7081.137248,2628.862752
5,15,6,7473.0,9709.910442,-2236.910442
6,37,6,5615.0,7473.145343,-1858.145343
7,7,6,8245.0,5614.618308,2630.381692
8,21,6,3192.0,8245.339971,-5053.339971
9,4,6,7882.0,3192.429750,4689.570250


Можно увидеть что результат в первой строке не изменился ведь неизвестно что получится если из настоящего вычесть неизвестное

Дальше я поменяю ORDER BY order_date внутри OVER на ORDER BY amount

In [4]:
pd.read_sql("""
    SELECT id, customer_id, ROUND(amount) AS amount,
           LAG(amount) OVER (PARTITION BY customer_id ORDER BY amount) AS prev,
           ROUND(amount) - LAG(amount) OVER (PARTITION BY customer_id ORDER BY amount) AS difference
    FROM orders WHERE customer_id = 6 ORDER BY order_date
""", conn)

,id,customer_id,amount,prev,difference
0,31,6,5580.0,4427.803997,1152.196003
1,24,6,2079.0,NaN,NaN
2,20,6,4428.0,3192.429750,1235.570250
3,26,6,7081.0,5614.618308,1466.381692
4,11,6,9710.0,8245.339971,1464.660029
5,15,6,7473.0,7081.137248,391.862752
6,37,6,5615.0,5580.436067,34.563933
7,7,6,8245.0,7882.036622,362.963378
8,21,6,3192.0,2079.091205,1112.908795
9,4,6,7882.0,7473.145343,408.854657


изменится теперь в prev будет лежать пердыдущий заказ по сумме, а у саммого дешевого будет лежать NULL